# Document Extraction and ChromaDB Embeddings

This notebook runs the repository's document extraction and embedding flow in Google Colab.
It stops after embeddings are stored and verified in ChromaDB. It does not run the application-analysis agents, semantic search, or RAG.

For restricted data, use synthetic documents only. Ollama is used locally in this Colab runtime for image and scanned-document OCR.

## 1. Clone the repository

In [ ]:
REPO_URL = "https://github.com/Govindkm/tcs-ai-club-hackathon-prompt-pioneers.git"
BRANCH = "develop"
PROJECT_DIR = "/content/tcs-ai-club-hackathon-prompt-pioneers"

import os

if not os.path.isdir(PROJECT_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {PROJECT_DIR}
else:
    print("Repository already exists.")

%cd {PROJECT_DIR}

## 2. Install required Python packages

This installs the repository dependencies, including `chromadb` and `sentence-transformers`.

In [ ]:
!pip install -q -r requirements.txt

## 3. Install Ollama

Ollama is required only when extraction needs OCR for images, scanned PDF pages, or embedded document images.

In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

## 4. Start Ollama and pull the OCR model

In [ ]:
import os
import subprocess
import time
import requests

OLLAMA_HOST = "http://127.0.0.1:11434"
VISION_MODEL_NAME = "minicpm-v"
os.environ["OLLAMA_HOST"] = OLLAMA_HOST
os.environ["OLLAMA_VISION_MODEL"] = VISION_MODEL_NAME

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

for attempt in range(30):
    try:
        response = requests.get(OLLAMA_HOST, timeout=2)
        if response.status_code == 200:
            print("Ollama server is running.")
            break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError("Ollama server did not start in time.")

!ollama pull {VISION_MODEL_NAME}

## 5. Configure local ChromaDB and embeddings

The multilingual Sentence Transformers model is downloaded and cached locally on its first use.

In [ ]:
import os

os.environ["CHROMA_PERSIST_DIRECTORY"] = "/content/chroma_db"
os.environ["CHROMA_COLLECTION_NAME"] = "scheme_documents"
os.environ["EMBEDDING_MODEL_NAME"] = (
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
os.environ["EMBEDDING_CHUNK_SIZE"] = "1000"
os.environ["EMBEDDING_CHUNK_OVERLAP"] = "150"
os.environ["EMBEDDING_BATCH_SIZE"] = "32"

print("Chroma path:", os.environ["CHROMA_PERSIST_DIRECTORY"])
print("Collection:", os.environ["CHROMA_COLLECTION_NAME"])
print("Embedding model:", os.environ["EMBEDDING_MODEL_NAME"])

## 6. Upload multiple documents

In [ ]:
from google.colab import files

uploaded_files = files.upload()
print(f"Uploaded files: {len(uploaded_files)}")

for filename, content in uploaded_files.items():
    print(f"{filename}: {len(content):,} bytes")

## 7. Extract text into structured document records

Each uploaded file becomes a logical document. ZIP entries retain their archive path.

In [ ]:
from src.ingestion.document_reader import extract_documents

file_payloads = list(uploaded_files.items())
documents = extract_documents(file_payloads)

print(f"Extracted logical documents: {len(documents)}")
for index, document in enumerate(documents, start=1):
    print(f"\nDocument {index}")
    print("Title:", document["title"])
    print("Extension:", document["extension"])
    print("Source path:", document["source_path"])
    print("Status:", document["status"])
    print("Content length:", len(document["content"]))
    print("Preview:", document["content"][:250])

## 8. Add optional pasted text as a document

Leave this cell unchanged if you are testing uploaded files only.

In [ ]:
PASTED_TEXT = ""
# Example:
# PASTED_TEXT = "Requested amount: Rs. 250,000. Project: solar micro-grid."

if PASTED_TEXT.strip():
    documents.append(
        {
            "title": "pasted-text.txt",
            "extension": ".txt",
            "source_path": "pasted-text.txt",
            "metadata": '{"source": "colab-pasted-text"}',
            "content": PASTED_TEXT.strip(),
            "status": "extracted",
            "error": None,
        }
    )

print("Documents ready for embedding:", len(documents))

## 9. Inspect the extracted manifest

In [ ]:
import json

print(json.dumps(documents, ensure_ascii=False, indent=2)[:5000])

## 10. Create the ChromaDB store

In [ ]:
from src.vectorstore.chroma_store import ChromaStore

chroma_store = ChromaStore()
print("ChromaDB store ready.")

## 11. Generate embeddings and store records

Use unique IDs for each test submission. The same submission can be safely re-indexed.

In [ ]:
SCHEME_ID = 1
SCHEME_TITLE = "Green Energy Grant"
SUBMISSION_ID = 1001

index_result = chroma_store.replace_documents(
    documents=documents,
    scheme_id=SCHEME_ID,
    scheme_title=SCHEME_TITLE,
    submission_id=SUBMISSION_ID,
)

print(json.dumps(index_result, indent=2))

## 12. Verify the number of Chroma records

In [ ]:
collection = chroma_store._get_collection()
actual_count = collection.count()
expected_count = index_result["indexed_chunk_count"]

print("Actual Chroma record count:", actual_count)
print("Expected Chroma record count:", expected_count)
assert actual_count == expected_count
print("Record count verification passed.")

## 13. Display stored records and metadata

In [ ]:
records = collection.get(include=["documents", "metadatas"])

for record_id, content, metadata in zip(
    records["ids"], records["documents"], records["metadatas"]
):
    print("=" * 80)
    print("Record ID:", record_id)
    print("Content preview:", content[:300])
    print("Metadata:")
    for key, value in metadata.items():
        print(f"  {key}: {value}")

## 14. Verify scheme and submission metadata

In [ ]:
submission_records = collection.get(
    where={"SubmissionID": str(SUBMISSION_ID)},
    include=["documents", "metadatas"],
)

assert len(submission_records["ids"]) == expected_count
for metadata in submission_records["metadatas"]:
    assert metadata["SchemeID"] == str(SCHEME_ID)
    assert metadata["SchemeTitle"] == SCHEME_TITLE
    assert metadata["SubmissionID"] == str(SUBMISSION_ID)

print("Metadata verification passed for submission", SUBMISSION_ID)

## 15. Verify persistence after reopening ChromaDB

In [ ]:
import chromadb

reopened_client = chromadb.PersistentClient(
    path=os.environ["CHROMA_PERSIST_DIRECTORY"]
)
reopened_collection = reopened_client.get_collection(
    os.environ["CHROMA_COLLECTION_NAME"]
)

print("Records after reopening:", reopened_collection.count())
assert reopened_collection.count() == expected_count
print("Persistence verification passed.")

## 16. Verify idempotent re-indexing

In [ ]:
second_result = chroma_store.replace_documents(
    documents=documents,
    scheme_id=SCHEME_ID,
    scheme_title=SCHEME_TITLE,
    submission_id=SUBMISSION_ID,
)

assert collection.count() == second_result["indexed_chunk_count"]
print("Idempotent re-indexing passed.")

## 17. Cleanup

Run this cell when finished. Colab storage is temporary, so the Chroma database will disappear when the runtime is deleted.

In [ ]:
if ollama_process.poll() is None:
    ollama_process.terminate()
    ollama_process.wait(timeout=10)
print("Ollama stopped.")